# $\arctan(1/\sqrt{15})/\sqrt{15}$: historical CMF bounds

Each section constructs a CMF, walks its trajectory, and prints $\delta_N$ at source depth 1000 for the reduced rational approximant. A finite-depth value does not prove the published measure bound.

In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

The CMFs are constructed directly by `ramanujantools.cmf.pfq.pFq`. The starting readouts, walks, and approximants are exact.

In [ ]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx

from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
with ctx.workdps(7000):
    TARGET = (arb(1)/arb(15).sqrt()).atan()/arb(15).sqrt()
DEPTHS = list(range(10, 1001, 10))
x0, x1, y0 = sp.symbols("x0 x1 y0")
results = {}


def delta(approximant):
    """Finite-depth exponent of the reduced rational approximant."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2*q.bit_length() + 128):
        error = abs(TARGET - arb(p)/q)
        assert error > 0
        return float((-1 - error.log()/arb(q).log()).mid())

## 1. Alladi–Robinson family: $\arctan(1/\sqrt{15})/\sqrt{15}$

[Quadratic integral and measure formula](https://www.math.ru.nl/~wzudilin/gat/oALLADI4.txt).
$$E_n=\int_0^1\frac{t^n(1-t)^n}{(t^2+15)^{n+1}}\,dt.$$
**CMF:** rank 2, ${}_2F_1(-1/15)$ ? **Start:** $(1/2,1;3/2)$ ? **Direction:** $(1,1;2)$. One step advances the source index by two; the plotted samples use $N/2$ steps.
With $c=\arctan(1/\sqrt{15})/\sqrt{15}$, $E_n=C_n+B_nc$ and $p_n/q_n=-C_n/B_n$. The source labels $c$ differently; the integral fixes its value. The displayed measure formula also uses its denominator lemma.
The exact identity
$$E_n=\frac{(n!)^2}{(2n+1)!\,15^{n+1}}{}_2F_1\!\left(\frac{n+1}{2},\frac{n+2}{2};n+\frac32;-\frac1{15}\right)$$
follows by expanding the quadratic denominator and cancelling an equal upper and lower parameter. The common scalar cancels in the rational readout.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 15))
start = {x0: sp.Rational(1, 2), x1: 1, y0: sp.Rational(3, 2)}
trajectory = {x0: 1, x1: 1, y0: 2}
# One lattice step advances the source index by two; all sampled N are even.
walks = cmf.walk(trajectory, [N//2 for N in DEPTHS], start)
approximants = [-1*M[1, 0]/(16*(2*M[0, 0]-M[1, 0])) for M in walks]

deltas = [delta(r) for r in approximants]
results["Alladi–Robinson family"] = (DEPTHS, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## 2. Hata (1992): specialization of Theorem 3.1

[Theorem 3.1 and Eq. (2.6), pp. 341-346](https://matwbn.icm.edu.pl/ksiazki/aa/aa60/aa6043.pdf)
apply with $(d,s,\lambda)=(1/2,-15,4)$ and $m=\lfloor n/4\rfloor$.
This gives $\mu(\alpha)\le 3.541128436736632948210954\ldots$ for $\alpha=\arctan(1/\sqrt{15})/\sqrt{15}$.
This decimal value is our specialization of the published theorem, not a numerical example printed in the paper.

After integration by parts, the remainder is a nonzero rational multiple of
$$J_0(n-\tfrac12,n+m,n+1)=
\int_0^1\frac{t^{n-1/2}(1-t)^{n+m}}{(1+t/15)^{n+1}}\,dt.$$
**CMF:** rank 2, ${}_2F_1(-1/15)$; **start:** $(1,1/2;3/2)$; **direction:** $(4,4;9)$.
The seed is source $n=0$; $k$ steps give $n=4k$, $m=k$, so 250 steps give $n=1000$.
In the $(F,\theta F)$ basis, use $C=(0,15/16)$ and $B=(2,-1)$.
The target readout is $-(CM)_0/[15(BM)_0]$.

**Theorem evaluation.** Let $g\in(0,1)$ and $h<-15$ solve $5x^2+915x-415=0$.
Put $L(x)=\log|x-1|+4\log|9x/4-1|$,
$\chi(k)=\sum_{j=1}^{k-1}\cot(j\pi/(2k))$, and
$\gamma=10-\log2-\frac{\pi}{2}(\chi(5)-\chi(4))$.
The source bound is $\kappa=1-[L(h)+\gamma]/[L(g)+\gamma]$.
The source congruence $-15\equiv1\pmod4$ holds, and interval arithmetic verifies
$L(g)+\gamma<0$.

The script local_checks/verify_hata_arctan.py checks both coefficient rows against
Hata's polynomial (1.2) and rational integral on p. 344 at $n=0,\ldots,8$,
and checks the fixed-direction walk at $n=0,4,8$.
The file local_checks/hata_arctan_checks.json records the theorem evaluation and two-precision
interval verification at $n=1000$. The source's arithmetic proof is not independently reproved here.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 15))
start = {x0: 1, x1: sp.Rational(1, 2), y0: sp.Rational(3, 2)}
trajectory = {x0: 4, x1: 4, y0: 9}
depths = list(range(12, 1000, 12)) + [1000]
walks = cmf.walk(trajectory, [N//4 for N in depths], start)
C = Matrix([[0, sp.Rational(15, 16)]])
B = Matrix([[2, -1]])
approximants = [-(C*M)[0]/(15*(B*M)[0]) for M in walks]

deltas = [delta(r) for r in approximants]
results["Hata (1992), theorem specialization"] = (depths, deltas)
print(f"delta at source depth 1000: {deltas[-1]:.12f}")


delta at source depth 1000: 0.398912671264


## All constructions

$\delta_N$ against the source depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published upper bound $\mu_0$ | $\delta_{\rm eq}$ | $\delta_{1000}$ | CMF |
|:--|--:|--:|--:|:--|
| Alladi–Robinson family | 3.880694268140 | 0.347138539157 | 0.355631015098 | rank-2 ${}_2F_1$ walk |
| Hata (1992), Theorem 3.1 specialization | 3.541128436737... | 0.393525957029 | 0.398912671264 | rank-2 ${}_2F_1$ walk |

$\delta_{\rm eq}=1/(\mu_0-1)$; the finite-depth values are diagnostics. The arithmetic denominator claims belong to the cited sources.